# Data Exploration

#### Pre Bronze Ingestion Checks

In [0]:
import os

tables = [
    "users", "orders", "order_items", "products", 
    "inventory_items", "distribution_centers", "events"
]

volume_path = "/Volumes/ecommerce/bronze/raw_data"

print(f"{'Table':<25} {'Rows':>10} {'Columns':>8}")
print("-" * 45)

for table in tables:
    path = f"{volume_path}/{table}.parquet"
    df = spark.read.parquet(path)
    print(f"{table:<25} {df.count():>10,} {len(df.columns):>8}")

Table                           Rows  Columns
---------------------------------------------
users                        100,000       16
orders                       124,850        9
order_items                  181,424       11
products                      29,120        9
inventory_items              490,083       12
distribution_centers              10        5
events                     2,427,889       13


In [0]:
df = spark.read.parquet(f"{volume_path}/users.parquet")
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- state: string (nullable = true)
 |-- street_address: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- traffic_source: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- user_geom: string (nullable = true)



In [0]:
tables = [
    "orders", "order_items", "products", 
    "inventory_items", "distribution_centers", "events"
]

volume_path = "/Volumes/ecommerce/bronze/raw_data"

for table in tables:
    df = spark.read.parquet(f"{volume_path}/{table}.parquet")
    print(f"\n{'='*60}")
    print(f"  {table.upper()}")
    print(f"{'='*60}")
    df.printSchema()


  ORDERS
root
 |-- order_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- status: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- returned_at: timestamp (nullable = true)
 |-- shipped_at: timestamp (nullable = true)
 |-- delivered_at: timestamp (nullable = true)
 |-- num_of_item: long (nullable = true)


  ORDER_ITEMS
root
 |-- id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- inventory_item_id: long (nullable = true)
 |-- status: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- shipped_at: timestamp (nullable = true)
 |-- delivered_at: timestamp (nullable = true)
 |-- returned_at: timestamp (nullable = true)
 |-- sale_price: double (nullable = true)


  PRODUCTS
root
 |-- id: long (nullable = true)
 |-- cost: double (nullable = true)
 |-- category: string (nullable = true)
 |-- n

#### Pre Silver Layer Checks

#### Null Profile Check

In [0]:
from pyspark.sql import functions as F

tables = {
    "ecommerce.bronze.users_raw": "id",
    "ecommerce.bronze.orders_raw": "order_id",
    "ecommerce.bronze.order_items_raw": "id",
    "ecommerce.bronze.products_raw": "id",
    "ecommerce.bronze.inventory_items_raw": "id",
    "ecommerce.bronze.distribution_centers_raw": "id",
    "ecommerce.bronze.events_raw": "id",
}

print("=" * 80)
print("  NULL PROFILE (single scan per table)")
print("=" * 80)

for table_name, pk in tables.items():
    short_name = table_name.split(".")[-1]
    df = spark.table(table_name)
    total = df.count()
    
    # Get string columns to check for literal "null" strings
    string_cols = [f.name for f in df.schema.fields if str(f.dataType) == "StringType"]
    
    # One scan for all null counts
    null_exprs = []
    for f in df.schema.fields:
        if f.name.startswith("_"):
            continue
        if f.name in string_cols:
            # Check both actual NULL and string "null"
            null_exprs.append(
                F.count(F.when(F.col(f.name).isNull() | (F.col(f.name) == "null"), True)).alias(f.name)
            )
        else:
            # Non-string columns: only check actual NULL
            null_exprs.append(
                F.count(F.when(F.col(f.name).isNull(), True)).alias(f.name)
            )
    
    null_counts = df.select(null_exprs).collect()[0]
    cols_with_nulls = {c: null_counts[c] for c in null_counts.asDict() if null_counts[c] > 0}
    
    print(f"\n  {short_name} ({total:,} rows)")
    if cols_with_nulls:
        for col, cnt in sorted(cols_with_nulls.items(), key=lambda x: -x[1]):
            print(f"    {col:<30} {cnt:>10,} nulls ({cnt/total*100:.1f}%)")
    else:
        print(f"    No nulls found")

  NULL PROFILE (single scan per table)

  users_raw (100,000 rows)
    No nulls found

  orders_raw (124,850 rows)
    returned_at                       112,468 nulls (90.1%)
    delivered_at                       81,366 nulls (65.2%)
    shipped_at                         43,948 nulls (35.2%)

  order_items_raw (181,424 rows)
    returned_at                       163,510 nulls (90.1%)
    delivered_at                      118,256 nulls (65.2%)
    shipped_at                         63,697 nulls (35.1%)

  products_raw (29,120 rows)
    brand                                  24 nulls (0.1%)
    name                                    2 nulls (0.0%)

  inventory_items_raw (490,083 rows)
    sold_at                           308,659 nulls (63.0%)

  distribution_centers_raw (10 rows)
    No nulls found

  events_raw (2,427,889 rows)
    user_id                         1,124,064 nulls (46.3%)


#### Foreign Key Integrity Check

In [0]:
print("=" * 80)
print("  FOREIGN KEY INTEGRITY")
print("=" * 80 + "\n")

fk_checks = [
    ("orders_raw",          "user_id",      "users_raw",    "id"),
    ("order_items_raw",     "order_id",     "orders_raw",   "order_id"),
    ("order_items_raw",     "user_id",      "users_raw",    "id"),
    ("order_items_raw",     "product_id",   "products_raw", "id"),
    ("inventory_items_raw", "product_id",   "products_raw", "id"),
    ("events_raw",          "user_id",      "users_raw",    "id"),
]

for child_table, fk_col, parent_table, pk_col in fk_checks:
    child = spark.table(f"ecommerce.bronze.{child_table}")
    parent = spark.table(f"ecommerce.bronze.{parent_table}")
    
    orphans = child.join(parent, child[fk_col] == parent[pk_col], "left_anti").count()
    total = child.count()
    
    status = "✓" if orphans == 0 else "⚠"
    print(f"  {status} {child_table}.{fk_col} → {parent_table}.{pk_col}: {orphans:,} orphans ({orphans/total*100:.2f}%)")

  FOREIGN KEY INTEGRITY

  ✓ orders_raw.user_id → users_raw.id: 0 orphans (0.00%)
  ✓ order_items_raw.order_id → orders_raw.order_id: 0 orphans (0.00%)
  ✓ order_items_raw.user_id → users_raw.id: 0 orphans (0.00%)
  ✓ order_items_raw.product_id → products_raw.id: 0 orphans (0.00%)
  ✓ inventory_items_raw.product_id → products_raw.id: 0 orphans (0.00%)
  ⚠ events_raw.user_id → users_raw.id: 1,124,064 orphans (46.30%)


#### String Cardinality Check

In [0]:
print("=" * 80)
print("  STRING CARDINALITY")
print("=" * 80 + "\n")

cardinality_checks = [
    ("users_raw",       ["country", "gender", "traffic_source"]),
    ("orders_raw",      ["status", "gender"]),
    ("order_items_raw", ["status"]),
    ("products_raw",    ["category", "department"]),
    ("events_raw",      ["event_type", "browser", "traffic_source"]),
]

for table_name, columns in cardinality_checks:
    df = spark.table(f"ecommerce.bronze.{table_name}")
    print(f"  {table_name}")
    for col in columns:
        distinct = df.select(col).distinct().count()
        print(f"    {col:<25} {distinct:>5} distinct values")
        df.groupBy(col).count().orderBy("count", ascending=False).show(10, truncate=False)

  STRING CARDINALITY

  users_raw
    country                      15 distinct values
+--------------+-----+
|country       |count|
+--------------+-----+
|China         |33907|
|United States |22511|
|Brasil        |14551|
|South Korea   |5387 |
|France        |4791 |
|United Kingdom|4615 |
|Germany       |4170 |
|Spain         |3974 |
|Japan         |2420 |
|Australia     |2176 |
+--------------+-----+
only showing top 10 rows
    gender                        2 distinct values
+------+-----+
|gender|count|
+------+-----+
|M     |50136|
|F     |49864|
+------+-----+

    traffic_source                5 distinct values
+--------------+-----+
|traffic_source|count|
+--------------+-----+
|Search        |69940|
|Organic       |15042|
|Facebook      |6095 |
|Email         |4876 |
|Display       |4047 |
+--------------+-----+

  orders_raw
    status                        5 distinct values
+----------+-----+
|status    |count|
+----------+-----+
|Shipped   |37418|
|Complete  |31102|
|Pro

#### Timestamp Lifecycle Validation Check

In [0]:
print("=" * 80)
print("  TIMESTAMP LIFECYCLE VALIDATION")
print("=" * 80 + "\n")

orders = spark.table("ecommerce.bronze.orders_raw")

print("Order status distribution:")
orders.groupBy("status").count().orderBy("count", ascending=False).show()

# Full lifecycle check
print("Timestamp violations:")
print(f"  shipped_at < created_at:   {orders.filter(F.col('shipped_at') < F.col('created_at')).count():,}")
print(f"  delivered_at < shipped_at: {orders.filter(F.col('delivered_at') < F.col('shipped_at')).count():,}")
print(f"  returned_at < delivered_at:{orders.filter(F.col('returned_at') < F.col('delivered_at')).count():,}")

print(f"\nDate range: ")
orders.select(F.min("created_at").alias("earliest"), F.max("created_at").alias("latest")).show(truncate=False)

# Order items lifecycle
oi = spark.table("ecommerce.bronze.order_items_raw")
print("Order Items timestamp violations:")
print(f"  delivered_at < shipped_at: {oi.filter(F.col('delivered_at') < F.col('shipped_at')).count():,}")
print(f"  returned_at < delivered_at:{oi.filter(F.col('returned_at') < F.col('delivered_at')).count():,}")

# Inventory lifecycle  
inv = spark.table("ecommerce.bronze.inventory_items_raw")
print(f"\nInventory: sold_at < created_at: {inv.filter(F.col('sold_at') < F.col('created_at')).count():,}")
print(f"Inventory: unsold items (sold_at is null): {inv.filter(F.col('sold_at').isNull()).count():,}")

  TIMESTAMP LIFECYCLE VALIDATION

Order status distribution:
+----------+-----+
|    status|count|
+----------+-----+
|   Shipped|37418|
|  Complete|31102|
|Processing|24840|
| Cancelled|19108|
|  Returned|12382|
+----------+-----+

Timestamp violations:
  shipped_at < created_at:   0
  delivered_at < shipped_at: 0
  returned_at < delivered_at:0

Date range: 
+-------------------+--------------------------+
|earliest           |latest                    |
+-------------------+--------------------------+
|2019-01-07 06:19:43|2026-07-16 00:47:44.727604|
+-------------------+--------------------------+

Order Items timestamp violations:
  delivered_at < shipped_at: 0
  returned_at < delivered_at:0

Inventory: sold_at < created_at: 0
Inventory: unsold items (sold_at is null): 308,659


#### Distributions

In [0]:
print("=" * 80)
print("  NUMERIC DISTRIBUTIONS")
print("=" * 80 + "\n")

# Users age
users = spark.table("ecommerce.bronze.users_raw")
print("Users — age:")
users.select(
    F.min("age"), F.max("age"), 
    F.avg("age").cast("int").alias("avg_age"),
    F.expr("percentile_approx(age, 0.01)").alias("p1"),
    F.expr("percentile_approx(age, 0.99)").alias("p99")
).show()

# Order items sale_price
oi = spark.table("ecommerce.bronze.order_items_raw")
print("Order Items — sale_price:")
oi.select(
    F.min("sale_price"), F.max("sale_price"),
    F.avg("sale_price").cast("decimal(10,2)").alias("avg"),
    F.expr("percentile_approx(sale_price, 0.01)").alias("p1"),
    F.expr("percentile_approx(sale_price, 0.99)").alias("p99"),
    F.count(F.when(F.col("sale_price") <= 0, True)).alias("zero_or_neg")
).show()

# Products margin check
prods = spark.table("ecommerce.bronze.products_raw")
print("Products — negative margin (cost > retail_price):")
neg = prods.filter(F.col("cost") > F.col("retail_price")).count()
print(f"  {neg} products with negative margin")

  NUMERIC DISTRIBUTIONS

Users — age:
+--------+--------+-------+---+---+
|min(age)|max(age)|avg_age| p1|p99|
+--------+--------+-------+---+---+
|      12|      70|     40| 12| 70|
+--------+--------+-------+---+---+

Order Items — sale_price:
+------------------+---------------+-----+---+-----+-----------+
|   min(sale_price)|max(sale_price)|  avg| p1|  p99|zero_or_neg|
+------------------+---------------+-----+---+-----+-----------+
|0.0199999995529651|          999.0|59.62|6.5|300.0|          0|
+------------------+---------------+-----+---+-----+-----------+

Products — negative margin (cost > retail_price):
  0 products with negative margin


#### FK & String Check

In [0]:
events = spark.table("ecommerce.bronze.events_raw").filter(F.col("user_id").isNotNull())
users = spark.table("ecommerce.bronze.users_raw")

true_orphans = events.join(users, events.user_id == users.id, "left_anti").count()
print(f"True orphan user_ids (non-null but missing from users): {true_orphans:,}")

True orphan user_ids (non-null but missing from users): 0


In [0]:
users = spark.table("ecommerce.bronze.users_raw")

actual_null = users.filter(F.col("city").isNull()).count()
string_null = users.filter(F.col("city") == "null").count()
print(f"city — actual NULL: {actual_null:,}")
print(f"city — string 'null': {string_null:,}")

city — actual NULL: 0
city — string 'null': 913


In [0]:
users = spark.table("ecommerce.bronze.users_raw")

string_cols = ["first_name", "last_name", "email", "gender", "state", 
               "street_address", "postal_code", "city", "country", 
               "traffic_source", "user_geom"]

print("String 'null' check across all users columns:")
for col in string_cols:
    cnt = users.filter(F.col(col) == "null").count()
    if cnt > 0:
        print(f"  {col:<25} {cnt:,} string 'null' values")

print("\nDone.")

String 'null' check across all users columns:
  city                      913 string 'null' values

Done.
